In [181]:
%pip install -U langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface chromadb sentence-transformers transformers accelerate torch ipykernel jupyter faiss-cpu

Note: you may need to restart the kernel to use updated packages.



THE FOLLOWING FEW CELLS WILL:

load merged_corpus.jsonl

convert rows into LangChain Documents

chunk them

embed them

save them into Chroma

In [244]:
#imports
import json
import re
from pathlib import Path
import os

# LangChain 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document as LCDocument

# Models
from sentence_transformers import CrossEncoder
from transformers import pipeline

In [245]:
#paths and config
CORPUS_PATH = Path("merged_corpus.jsonl")
CHROMA_DIR = "chroma_db"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

INITIAL_RETRIEVAL_K = 10
FINAL_K = 5
MAX_NEW_TOKENS = 120

In [246]:
def corpus_to_documents(corpus_rows):
    docs = []

    for i, r in enumerate(corpus_rows):
        text = r.get("text", "").strip()

        if not text:
            continue

        docs.append(
            LCDocument(
                page_content=text,
                metadata={
                    "doc_id": str(i),
                    "source": r.get("source", ""),
                    "title": r.get("title", ""),
                },
            )
        )

    return docs

def load_jsonl(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows

rows = load_jsonl(CORPUS_PATH)
docs = corpus_to_documents(rows)

In [256]:
#chunk the documents
# RecursiveCharacterTextSplitter - LangChain for text splitting.

def split_documents(docs, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    split_docs = splitter.split_documents(docs)

    # Add chunk IDs
    per_doc_counts = {}
    out = []

    for d in split_docs:
        doc_id = d.metadata.get("doc_id", "unknown")

        per_doc_counts[doc_id] = per_doc_counts.get(doc_id, 0) + 1
        idx = per_doc_counts[doc_id] - 1

        md = dict(d.metadata)
        md["chunk_id"] = f"{doc_id}::c{idx:05d}"

        out.append(LCDocument(page_content=d.page_content, metadata=md))

    return out

chunks = split_documents(docs, 500, 100)

In [257]:
#create embeddings


def build_embeddings(model_name="BAAI/bge-small-en-v1.5", device=None):
    model_kwargs = {}
    if device:
        model_kwargs["device"] = device

    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs
    )

embedding = build_embeddings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [258]:
import os

def save_vectorstore(vs, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    vs.save_local(out_dir)

def load_vectorstore(out_dir, embedding):
    return FAISS.load_local(
        out_dir,
        embedding,
        allow_dangerous_deserialization=True
    )

# Build once
vectorstore = FAISS.from_documents(chunks, embedding)

save_vectorstore(vectorstore, "faiss_store")

# OR load (instead of rebuilding)
vectorstore = load_vectorstore("faiss_store", embedding)

def retrieve_docs(query):
    return vectorstore.max_marginal_relevance_search(
        query,
        k=5,
        fetch_k=10
    )

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs):
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    return [doc for doc, _ in ranked[:5]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [276]:
def extract_ingredients(contexts):
    text = " ".join(contexts).lower()

    if "ingredients such as" in text:
        part = text.split("ingredients such as")[1]
        part = part.split(".")[0]

        return part.strip()

    return None

def is_recipe_query(query):
    keywords = ["recipe", "how to make", "how is", "steps", "method"]
    return any(k in query.lower() for k in keywords)

def extract_recipe(query, contexts):
    text = contexts[0]

    steps = []

    for line in text.split("\n"):
        line_lower = line.lower().strip()

        # skip empty lines
        if not line_lower:
            continue

        if not any(word in line_lower for word in ["mix", "add", "bake", "pour", "arrange", "divide", "brush", "cut"]):
            # if we already started collecting → stop
            if steps:
                break
            continue

        steps.append(line.strip())

    return "\n".join(steps) if steps else None

In [277]:
def build_prompt(query, contexts):
    context_text = "\n\n".join(contexts)

    return f"""
    You are an expert Mediterranean culinary assistant.
    
    Answer the question using ONLY the context below.
    
    Rules:
    - Do NOT add new information
    - Keep the answer concise
    - Only use what is written in the context
    
    Context:
    {context_text}
    
    Question: {query}
    Answer:
    """

In [278]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=150
)


def generate_answer(prompt):
    result = generator(prompt)
    return result[0]["generated_text"]

def filter_answer(answer, contexts):
    context_text = " ".join(contexts).lower()
    filtered_items = []

    for item in answer.split(","):
        item_clean = item.strip().lower()

        if item_clean in context_text:
            filtered_items.append(item.strip())

    return ", ".join(filtered_items)

def select_best_context(query, contexts):
    query_terms = query.lower().split()

    scored = []

    for c in contexts:
        score = sum(term in c.lower() for term in query_terms)
        scored.append((c, score))

    # sort by relevance
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    return scored[0][0] if scored else None

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [279]:
def rag_pipeline(query):
    retrieved = retrieve_docs(query)
    reranked = rerank(query, retrieved)

    contexts = [doc.page_content for doc in reranked]

    # ingredients
    if "ingredient" in query.lower():
        extracted = extract_ingredients(contexts)
        if extracted:
            return extracted

    # recipes
    if is_recipe_query(query):
        best_context = select_best_context(query, contexts)

        if best_context:
            recipe = extract_recipe(query, [best_context])
            if recipe:
                return recipe

    # fallback
    prompt = build_prompt(query, contexts)
    raw_answer = generate_answer(prompt)
    return filter_answer(raw_answer, contexts)

In [280]:
def simple_f1(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)

    return 2 * (precision * recall) / (precision + recall)

In [281]:
query = "What ingredients are commonly used in Mediterranean cuisine?"

answer = rag_pipeline(query)

print(answer)

bread and wine, fresh herbs and fruit, olive oil, garlic, tomatoes, peppers, onions, fish and shellfish, rice, pasta, sausages, lentils, chickpeas, and nuts including hazelnuts, almonds, and pine nuts


In [ ]:
while True:
    query = input("\nEnter your question (or type 'exit' to quit): ").strip()

    if query.lower() == "exit":
        print("Goodbye.")
        break

    if not query:
        print("Please enter a question.")
        continue


    answer = rag_pipeline(query)

    print("\nANSWER:\n")
    print(answer)




Enter your question (or type 'exit' to quit):  recipe for baklava


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:

being careful not to form a paste.
Melt the butter in a saucepan over medium heat, then set aside.
Line a baking pan with parchment paper.
Count the number of sheets of phyllo dough and divide it by eight. Arrange one-eighth of the sheets in the bottom of the pan, on the parchment paper.
Divide the chopped nuts into seven equally sized portions.
Brush the phyllo dough with melted butter, and evenly distribute one portion of chopped nuts over the dough.
Arrange another eighth of the phyllo sheets on top of the nut layer.
Repeat the buttering and layering of nuts and phyllo until you have used up all the phyllo sheets and chopped nuts.
Using a sharp knife, radially cut the baklava into eight equally sized portions shaped like the British flag.
Bake the baklava for 50 minutes at 350°F (180°C). Remove from the oven, and let cool slightly.
Mix the vanilla extract, honey, sugar, and water in a clean saucepan. Boil over medium heat until it becomes a very thick and viscous syrup of 